In [2]:
from langchain_community.document_loaders import PyPDFLoader 
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
import os 
from dotenv import load_dotenv

load_dotenv()

/var/folders/k7/105m3s4d37s3qzhbkyh802mc0000gn/T/ipykernel_3763/3106893056.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


True

In [3]:
print("Loading PDF into text...")
loader = PyPDFLoader("/Users/rahulagarwal/Documents/ai practice/data/Aurelius_Robotics_Corporation_ARC_Sample_RAG_Document.pdf")
text_data = loader.load()
print("PDF loaded into text.")

Loading PDF into text...
PDF loaded into text.


In [4]:
# create chunks of text data
print("Creating chunks of text data...")
text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
text_chunks = text_splitter.split_documents(text_data)
print("Chunks of text data created.")
print(f"Number of text chunks created: {len(text_chunks)}")

Creating chunks of text data...
Chunks of text data created.
Number of text chunks created: 16


### create embeddings for the text chunks

In [5]:

# print("Creating embeddings for the text chunks...")
embedding_model = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2"
)

# embedded_chunks = embedding_model.embed_documents([chunk.page_content for chunk in text_chunks])

# print("Embeddings for the text chunks created.")


In [6]:
# print(f"Number of embedded chunks created: {len(embedded_chunks)}")
# print(f"First embedded chunk: {embedded_chunks[0]}")

## you dont have to create embeddings manually like in previous step. 
## embeddings + vector together as below. 

In [7]:
from langchain_community.vectorstores import Chroma


In [8]:
# (behind is a loop that creates embeddings for each chunk and saves it to disk) 
# This is done so that you dont have to re-embed the documents every time you run the code.
try:
    chroma_db = Chroma.from_documents(text_chunks, embedding_model, persist_directory="./vector_db")
except Exception as e:
    print(f"Error creating Chroma database: {e}")

In [9]:
chroma_db_con = Chroma(persist_directory="./vector_db", embedding_function=embedding_model)

/var/folders/k7/105m3s4d37s3qzhbkyh802mc0000gn/T/ipykernel_3763/3567316335.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  chroma_db_con = Chroma(persist_directory="./vector_db", embedding_function=embedding_model)


In [10]:
#chroma_db_con.similarity_search("what is the location of its factory?", k=3)

## LLM integration and answer generation

In [11]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os

llm = ChatGoogleGenerativeAI(
    model=os.getenv("GEMINI_MODEL"),
    temperature=0
)

user_query = "what is the valuation of reliance industries?"
relevant_chunks = chroma_db_con.similarity_search(user_query, k=3)
relevant_chunks_content = []
for i,chunks in enumerate(relevant_chunks):
    relevant_chunks_content.append(chunks.page_content)
    print(f"Relevant chunk {i}: {chunks.page_content}")


Relevant chunk 0: Aurelius Robotics Corporation (ARC)
Company Information
Founded: 2017
Headquarters: Bengaluru, India
Employees: Approximately 1,850
CEO: Dr. Maya Srinivasan
CTO: Daniel Brooks
Annual Revenue (2025): ■1,240 crore
Company Overview
Relevant chunk 1: Aurelius Robotics Corporation (ARC)
Company Information
Founded: 2017
Headquarters: Bengaluru, India
Employees: Approximately 1,850
CEO: Dr. Maya Srinivasan
CTO: Daniel Brooks
Annual Revenue (2025): ■1,240 crore
Company Overview
Relevant chunk 2: Aurelius Robotics Corporation (ARC)
Company Information
Founded: 2017
Headquarters: Bengaluru, India
Employees: Approximately 1,850
CEO: Dr. Maya Srinivasan
CTO: Daniel Brooks
Annual Revenue (2025): ■1,240 crore
Company Overview


In [12]:

relevant_chunks_content = str(relevant_chunks_content)
response = llm.invoke(f'{user_query},context={relevant_chunks_content}')
print(response.content)

[{'type': 'text', 'text': 'Based on the context provided, there is no information regarding the valuation of **Reliance Industries**.\n\nThe provided text focuses exclusively on **Aurelius Robotics Corporation (ARC)**, which has an annual revenue of ₹1,240 crore (as of 2025).', 'extras': {'signature': 'ErwNCrkNAQw51sfGvbC1xUz9M8SjRXMW/IXQr6H963moESdozpI52uL/P52CoWonkF5jwsH6xMX74otHdaUmSx37G2s6VDiKwg/GJsNTdmYfj1OrfivldDoMh4fHIOjdw8Js7NZsOz1GzNv2mA57eTDvZqNLDYMffovshJBr+g5YgpORCfVQIL4iGyU68/WhJY5im5DJyXq9WSIYmFWykqrO8ZBKAduq7Gdz1qdZik0xpq0QfE5eKI0J3WV6Yq0ikfQ7J0JmpnWGpDkx77WK5FgIeARNKi5vYnLwrpRTVn+GXN7stKq0hCGkQ1dpXGK7pfvveEyLFmzkgN5Ss7NN3tHQQW8++zsyLf62nFUqys5YYspIipWmg7eXszzWR4P0b0jn5xjwiWH/V4tIlttWgARm0DJUEhaFdYgHB9cKz54KW0gDfjVPnlGVEcr6PMrBSWX+GseMgb16x5MrEsejXouAJsYGmm50nYbDbzrLt4bWwXtCRR1klj4PtmTlYTR8vWWaN625xZpLP6Z9+65kAPML8ZkFfPSCvzJ2RRBnmlW9DHH7du2tOeNMAVHefbguEOZVCBHMfawiFfEd331sNP6aIOjPDlBwzHaouDiwbXHFPhZNpdlmLffCElrYhfL8jfLWyOF7G87O3opZclIvohbRN4yONNZFybyKMHI36NuSgNsNflVPpXpG

In [15]:
print(response.content[0]['text'])

Based on the context provided, there is no information regarding the valuation of **Reliance Industries**.

The provided text focuses exclusively on **Aurelius Robotics Corporation (ARC)**, which has an annual revenue of ₹1,240 crore (as of 2025).
